### This notebook compute "P13. Water yield: Water production per COMID and per square kilometer" indicator for the 27 basins of IKI Project

**Created:** 12/17/2025 by Sophia Bakar (sbakar@rti.org)

**Project #:** 0219481  

**Last modified:** 12/22/2025 by Sophia Bakar

**Status:** Complete for baseline scenario and first future scenario

**QA Status:** reviewed by  

**Original Script Stored at:** Research Triangle Institute\IKI Peru Project - General\Interno\AI2a_Metodologia\Indicadores\Peligro
 
**Objective:**   

**Compatibility:** 

**Packages:** numpy, pandas, geopandas, sqlite3

**Further documentation:**  
 
**Inputs:** area of each COMID (source: subbasins shapfile); annual average of water production for each COMID (source: waterALLOC output database)

**Outputs:** 
 
**Assumptions:** We do not normalize the results in this script because we normalize them as the first step of the impact chains calculation.

**Future work:** 
 
**Notes:** For this indicator, we get the annual average of water production for each COMID from the waterALLOC output database. Then we divide that by the area in square kilometers for each COMID. When we do the normalization step, we need to normalize by the min and max across all the COMIDs. In this script, we include code to check the min and max of our results and the min and max that is listed in the indicators database to ensure they align

In [1]:
import numpy as np
import pandas as pd
import sqlite3
import geopandas as gpd
import os
import re

In [2]:
user = 'sbakar'
#db_path = fr'C:\Users\{user}\Research Triangle Institute\IKI Peru Project - General\Interno\AI2a_Metodologia\Indicadores\BD_RiesgoClimatico_IKI.db'
db_path = fr"C:\Users\sbakar\OneDrive - Research Triangle Institute\IKI Peru Project - General\Interno\AI2a_Metodologia\Indicadores\BD_RiesgoClimatico_IKI.db"
# wateralloc_db = fr"C:\Users\{user}\Research Triangle Institute\IKI Peru Project - General\Interno\AI2b_Modelacion\Grupos_Modelacion\Resultados\BalanceHidrico.sqlite"
wateralloc_db = fr"C:\Users\sbakar\OneDrive - Research Triangle Institute\IKI Peru Project - General\Interno\AI2b_Modelacion\Grupos_Modelacion\Resultados\BalanceHidrico.sqlite"

In [3]:
# Set up indicator ID
IndID = 113  # Indicator ID (Exposure = 2 + 0X, Peligro = 1 +0x, etc.)

# Connect to Indicators DB and get available scenarios
conn = sqlite3.connect(db_path)

scenarios_df = pd.read_sql_query(
    """
    SELECT ScnID, ScnName
    FROM ScnMod
    ORDER BY ScnID
    """,
    conn
)

conn.close()

# For now: only baseline and first future
scenario_ids = scenarios_df.loc[
    scenarios_df['ScnID'].isin([1, 2]), 'ScnID'
].tolist()

# for all scenarios:
# scenario_ids = scenarios_df['ScnID'].tolist()

In [4]:
#subbasins_shapefile = f'C:/Users/{user}/Research Triangle Institute/IKI Peru Project - General/Interno/AI2b_Modelacion/Grupos_Modelacion/GIS_WaterALLOC_General/Peru_AHD_with_districts.shp'
subbasins_shapefile = fr"C:\Users\sbakar\OneDrive - Research Triangle Institute\IKI Peru Project - General\Interno\AI2b_Modelacion\Grupos_Modelacion\GIS_WaterALLOC_General\Peru_AHD_with_districts.shp"
subbasins_gdf = gpd.read_file(subbasins_shapefile).set_index('COMID').to_crs('WGS84')

In [6]:
## check what scenarios are available in the WaterALLOC database
# Connect to WaterALLOC database
conn_wa = sqlite3.connect(wateralloc_db)

# Query available scenarios
scenarios_query = """
SELECT DISTINCT Scenario
FROM Scenarios
ORDER BY Scenario
"""

wa_scenarios_df = pd.read_sql_query(scenarios_query, conn_wa)

print("Available scenarios in WaterALLOC DB:")
for s in wa_scenarios_df["Scenario"]:
    print(f" - {s}")



Available scenarios in WaterALLOC DB:
 - CC_CMIP6_85_2050
 - Linea_Base_2020


In [7]:
# Extract year from scenario name and sort by year
def extract_year(name):
    match = re.search(r'\d{4}$', name)
    if match:
        return int(match.group())
    else:
        return np.nan  # just in case

wa_scenarios_df['Year'] = wa_scenarios_df['Scenario'].apply(extract_year)
wa_scenarios_df = wa_scenarios_df.sort_values('Year').reset_index(drop=True)

# Assign dynamic ScnID based on year order
wa_scenarios_df['ScnID_dynamic'] = np.arange(1, len(wa_scenarios_df) + 1)

# Map dynamic ScnID -> scenario name
scenario_mapping = dict(zip(wa_scenarios_df['ScnID_dynamic'], wa_scenarios_df['Scenario']))

print("\nDynamic scenario mapping (ScnID -> WaterALLOC Scenario Name):")
for scn_id, scn_name in scenario_mapping.items():
    print(f" - ScnID {scn_id} -> {scn_name}")


Dynamic scenario mapping (ScnID -> WaterALLOC Scenario Name):
 - ScnID 1 -> Linea_Base_2020
 - ScnID 2 -> CC_CMIP6_85_2050


In [8]:
# Loop through all scenarios and fetch OfertaTot data
oferta_all = []

conn_wa = sqlite3.connect(wateralloc_db)
for scn_id_dynamic, scenario_name in scenario_mapping.items():
    print(f"\nProcessing scenario: {scenario_name} (ScnID={scn_id_dynamic})")

    query_oferta = """
    SELECT 
        a.comid AS COMID,
        AVG(a.[OfertaTot]) AS OfertaTot_mean,
        COUNT(a.[OfertaTot]) AS n_years
    FROM [WAMSS_Oferta anual por microcuenca] AS a
    JOIN WAMMS_RunsInfo AS b 
        ON a.RunID = b.RunID
    JOIN Scenarios AS c 
        ON c.ScnID = b.ScnID
    WHERE c.Scenario = ?
    GROUP BY a.comid
    """
    oferta_df = pd.read_sql_query(query_oferta, conn_wa, params=(scenario_name,))

    # Add dynamic ScnID column
    oferta_df['ScnID_dynamic'] = scn_id_dynamic

    # Merge basin area
    area_df = subbasins_gdf[['AREASQKM']].reset_index().rename(columns={'AREASQKM': 'Area_km2'})
    oferta_df = oferta_df.merge(area_df, on='COMID', how='left')

    # Normalize by area
    oferta_df['OfertaTot_mean_norm'] = oferta_df['OfertaTot_mean'] / oferta_df['Area_km2']

    oferta_all.append(oferta_df)
conn_wa.close()

oferta_all_df = pd.concat(oferta_all, ignore_index=True)


Processing scenario: Linea_Base_2020 (ScnID=1)

Processing scenario: CC_CMIP6_85_2050 (ScnID=2)


In [9]:
# Connect to SQLite database
conn = sqlite3.connect(db_path)
cursor = conn.cursor()

rows_to_insert = []
for _, row in oferta_all_df.iterrows():
    rows_to_insert.append((row['ScnID_dynamic'], IndID, row['COMID'], row['OfertaTot_mean_norm']))

# Insert data into IndValues_Dyn
insert_query = """
INSERT OR REPLACE INTO IndValues_Dyn (ScnID, IndID, COMID, Value)
VALUES (?, ?, ?, ?);
"""

In [10]:
# Check that min and max values match the expected range based on the Indicators Table

indicator_limits = pd.read_sql_query(
    """
    SELECT IndID, Min, Max
    FROM Indicators
    WHERE IndID = ?
    """,
    conn,
    params=(IndID,)
)

if indicator_limits.empty:
    raise ValueError(f"No entry found in Indicators table for IndID = {IndID}")

ind_min = indicator_limits.loc[0, 'Min']
ind_max = indicator_limits.loc[0, 'Max']

print(f"\nIndicator {IndID} limits from Indicators table -> Min: {ind_min}, Max: {ind_max}")

# Compute value stats by scenario
value_stats = oferta_all_df.groupby('ScnID_dynamic')['OfertaTot_mean_norm'].agg(['min', 'max', 'count']).reset_index()
print("\n=== Values to be inserted (by scenario) ===")
print(value_stats)

# Check for duplicates in the rows to insert
df_check = pd.DataFrame(rows_to_insert, columns=['ScnID', 'IndID', 'COMID', 'Value'])
duplicates = df_check.duplicated(subset=['ScnID', 'IndID', 'COMID'])
print("\nDuplicates in rows_to_insert:")
print(df_check[duplicates])



Indicator 113 limits from Indicators table -> Min: 1, Max: 4

=== Values to be inserted (by scenario) ===
   ScnID_dynamic         min          max  count
0              1  172.741412   944.699798     62
1              2  301.539412  1415.644426     62

Duplicates in rows_to_insert:
Empty DataFrame
Columns: [ScnID, IndID, COMID, Value]
Index: []


In [11]:
#Execute insert to SQLite Database
cursor.executemany(insert_query, rows_to_insert)
conn.commit()
conn.close()